In [2]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Primary balance vs previous-year debt, % of GDP. Source: OBR databank.
xl = "../Data/PSF_aggregates_databank_Mar_EFO.xlsx"   # adjust path
raw = pd.read_excel(xl, sheet_name="Aggregates (per cent of GDP)", header=None)
d = raw.iloc[4:, [1, 11, 23]].copy()
d.columns = ["year", "pb", "debt"]
d["year"] = d["year"].apply(lambda v: int(str(v)[:4]) if str(v)[:4].isdigit() else np.nan)
for c in ["pb", "debt"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.dropna().sort_values("year")
d = d[d["year"] <= 2024]
d["debt_lag"] = d["debt"].shift(1)
d = d.dropna()

def era(y):
    if y <= 1999: return "1975–1999"
    return "2000–2024"
d["era"] = d["year"].apply(era)
order = ["1975–1999", "2000–2024"]

pts = alt.Chart(d).mark_point(filled=True, size=60, opacity=0.75).encode(
    x=alt.X("debt_lag:Q", title="Debt (% of GDP), previous year"),
    y=alt.Y("pb:Q", title="Primary balance (% of GDP)"),
    color=alt.Color("era:N", sort=order,
        scale=alt.Scale(domain=order, range=["#36B7B4", "#E6224B"]),
        legend=alt.Legend(orient="top", title=None)))

fits = pts.transform_regression("debt_lag", "pb", groupby=["era"]).mark_line(strokeWidth=2.5)

zero = alt.Chart(d).mark_rule(color="#c9c9c9", strokeDash=[2,3]).encode(y=alt.datum(0))

caption = alt.Title(
    text="Source: author's calculation; OBR public finances databank",
    subtitle=[
        "UK primary balance vs previous-year debt, by period.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (zero + pts + fits).properties(width=700, height=270, title=caption).configure(
    background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="deficit_vs_debt", svg=True)
chart.save("deficit_vs_debt.png", scale_factor=2.0)
chart

alt.LayerChart(...)